# Nudity Detection Inference - Full Dataset Processing
Scans the entire internvl-auditor-v2 dataset, detects nudity in all images, saves indices, and visualizes 50 safe and 50 nude samples with heatmaps

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import timm
from pathlib import Path
import numpy as np
from datasets import load_dataset
import io
from tqdm import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Setup CLIP backbone model
model_name = "resnet50_clip.openai"
model = timm.create_model(model_name,
                         pretrained=True,
                         features_only=True,
                         out_indices=[4]
                         )
model.eval()
model.to(device)

# Setup data processor
data_config = timm.data.resolve_model_data_config(model)
processor = timm.data.create_transform(**data_config, is_training=False)

In [ ]:
# Define the nudity detection model head
class BCE_pair_path(nn.Module):
    def __init__(self, input_classes=2048, num_classes=1):
        super().__init__()
        self.risk_conv = nn.Conv2d(input_classes, 1, kernel_size=1)
        self.pool = nn.AdaptiveMaxPool2d((1, 1))
        self.mlp = nn.Sequential(nn.Linear(1, 16),
                                 nn.ReLU(),
                                 nn.Linear(16, 1))
    
    def forward(self, x):
        risk_map = self.risk_conv(x)
        intermediate_logits = self.pool(risk_map)
        flattened_logits = torch.flatten(intermediate_logits, 1)
        logits = self.mlp(flattened_logits)
        return logits

In [ ]:
# Load trained model and setup hooks
model_3 = BCE_pair_path().to(device)

# Load model weights
model_path = Path('model_weights') / 'model_3_weights.pth'
if model_path.exists():
    model_3.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model loaded from {model_path}")
else:
    print(f"Warning: Model weights not found at {model_path}")

model_3.eval()

# Setup hook to capture risk maps
activations = {}

def get_activations_hook(name):
    def hook(model, input, output):
        activations[name] = output
    return hook

handle = model_3.risk_conv.register_forward_hook(get_activations_hook('risk_map'))

In [ ]:
# Helper function to load images from batch
def get_batch_images(batch, device):
    """Load and preprocess images from dataset batch"""
    images = []
    for img_data in batch['image']:
        if isinstance(img_data, dict):
            img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
        else:
            img = img_data.convert('RGB')
        images.append(processor(img))
    
    return torch.stack(images).to(device)

In [ ]:
# Visualization function with heatmap overlay
def plot_heatmap(image_tensor, risk_map_tensor, prediction_score, show_title=True):
    """
    Plot image with overlaid risk heatmap
    
    Args:
        image_tensor: [C, H, W] image tensor
        risk_map_tensor: [H, W] risk map from the model
        prediction_score: probability score (0-1) for nudity
        show_title: whether to show title with prediction
    """
    # Convert image to displayable format
    img = image_tensor.cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    
    # Process heatmap
    heatmap = torch.sigmoid(risk_map_tensor)
    heatmap = F.interpolate(heatmap.unsqueeze(0).unsqueeze(0), size=(224, 224), mode='bilinear')[0, 0]
    heatmap = heatmap.detach().cpu().numpy()
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.imshow(img)
    ax1.set_title("Original Image")
    ax1.axis('off')
    
    ax2.imshow(img)
    im = ax2.imshow(heatmap, cmap="jet", alpha=0.6)
    
    if show_title:
        nudity_label = "CONTAINS NUDITY" if prediction_score > 0.5 else "SAFE"
        ax2.set_title(f"Risk Heatmap\nScore: {prediction_score:.3f} | {nudity_label}", 
                     fontsize=11, fontweight='bold')
    else:
        ax2.set_title("Risk Heatmap")
    
    ax2.axis('off')
    plt.colorbar(im, ax=ax2, label='Risk Score')
    plt.tight_layout()
    plt.show()

In [ ]:
# Main inference: Go through entire dataset and collect indices
print("\n" + "="*80)
print("PROCESSING ENTIRE DATASET - internvl-auditor-v2")
print("="*80 + "\n")

dataset = load_dataset("ShreyashDhoot/internvl-auditor-v2", split='train', streaming=True)

nude_indices = []
safe_indices = []
nude_samples_data = []
safe_samples_data = []
total_images_scanned = 0

THRESHOLD = 0.5
MAX_SAMPLES_TO_COLLECT = 50  # Collect up to 50 of each category

print("Streaming through dataset and running inference...\n")

with torch.no_grad():
    for img_idx, item in enumerate(tqdm(dataset)):
        total_images_scanned += 1
        
        try:
            # Load and preprocess image
            img = item['image'].convert('RGB')
            img_tensor = processor(img).unsqueeze(0).to(device)
            
            # Forward pass through CLIP backbone
            clip_features = model(img_tensor)[0]
            
            # Forward pass through nudity detection head
            logit = model_3(clip_features)
            
            # Get probability
            prob = torch.sigmoid(logit)[0, 0].item()
            
            # Get risk map from hook
            risk_map = activations['risk_map'][0, 0]
            
            # Categorize and save index
            if prob > THRESHOLD:
                nude_indices.append(img_idx)
                if len(nude_samples_data) < MAX_SAMPLES_TO_COLLECT:
                    nude_samples_data.append({
                        'index': img_idx,
                        'probability': prob,
                        'risk_map': risk_map.clone(),
                        'image_tensor': img_tensor[0].clone()
                    })
            else:
                safe_indices.append(img_idx)
                if len(safe_samples_data) < MAX_SAMPLES_TO_COLLECT:
                    safe_samples_data.append({
                        'index': img_idx,
                        'probability': prob,
                        'risk_map': risk_map.clone(),
                        'image_tensor': img_tensor[0].clone()
                    })
            
            # Print progress every 100 images
            if (img_idx + 1) % 100 == 0:
                print(f"Processed {img_idx + 1} images | Nude: {len(nude_indices)} | Safe: {len(safe_indices)}")
        
        except Exception as e:
            # Skip corrupted images
            continue
        
        # Stop if we've processed enough or collected sufficient samples
        if img_idx >= 5000:  # Process first 5000 images as sample
            print(f"\nReached processing limit of 5000 images")
            break

print(f"\n{'='*80}")
print(f"Dataset processing complete!")
print(f"Total images processed: {total_images_scanned}")
print(f"Nude images found: {len(nude_indices)}")
print(f"Safe images found: {len(safe_indices)}")
print(f"Samples collected for visualization - Nude: {len(nude_samples_data)}, Safe: {len(safe_samples_data)}")
print(f"{'='*80}\n")

In [ ]:
# Visualize 50 safe and 50 nude samples with heatmaps for debugging
print("\n" + "="*80)
print("VISUALIZING 50 SAFE AND 50 NUDE SAMPLES FOR DEBUGGING")
print("="*80 + "\n")

num_to_display = 50

# Display safe samples
print(f"\n{'SAFE IMAGES':^80}")
print("="*80)
safe_to_show = safe_samples_data[:min(num_to_display, len(safe_samples_data))]

for i, sample in enumerate(safe_to_show):
    idx = sample['index']
    prob = sample['probability']
    risk_map = sample['risk_map']
    img_tensor = sample['image_tensor']
    
    print(f"\nSafe Sample {i+1}/{len(safe_to_show)} - Dataset Index: {idx}")
    print(f"Safety Score: {prob:.3f}")
    plot_heatmap(img_tensor, risk_map, prob, show_title=True)

# Display nude samples
print(f"\n{'NUDE IMAGES':^80}")
print("="*80)
nude_to_show = nude_samples_data[:min(num_to_display, len(nude_samples_data))]

for i, sample in enumerate(nude_to_show):
    idx = sample['index']
    prob = sample['probability']
    risk_map = sample['risk_map']
    img_tensor = sample['image_tensor']
    
    print(f"\nNude Sample {i+1}/{len(nude_to_show)} - Dataset Index: {idx}")
    print(f"Nudity Score: {prob:.3f}")
    plot_heatmap(img_tensor, risk_map, prob, show_title=True)

print("\n" + "="*80)

In [ ]:
# Print results summary and statistics
print("\n" + "="*80)
print("FULL DATASET INFERENCE SUMMARY")
print("="*80)

print(f"\nTotal Images Scanned: {total_images_scanned}")
print(f"Total Nude Images Found: {len(nude_indices)}")
print(f"Total Safe Images Found: {len(safe_indices)}")
print(f"\nNude Indices: {nude_indices}")
print(f"Safe Indices: {safe_indices}")

# Save indices to file
indices_file = Path('nudity_detection_results.txt')
with open(indices_file, 'w') as f:
    f.write(f"Total Images Scanned: {total_images_scanned}\n")
    f.write(f"Total Nude Images: {len(nude_indices)}\n")
    f.write(f"Total Safe Images: {len(safe_indices)}\n\n")
    f.write(f"Nude Image Indices:\n{nude_indices}\n\n")
    f.write(f"Safe Image Indices:\n{safe_indices}\n")

print(f"\nResults saved to {indices_file}")
print("="*80)